# 03 · Indexing and Selection

**Goal:** master the three main ways to select data in pandas: label-based (`.loc`),
position-based (`.iloc`), and boolean filtering — the single most important skill in pandas.

### Setup: a sample DataFrame with a meaningful index

In [1]:
import pandas as pd

df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana", "Evan"],
    "age": [25, 32, 18, 47, 29],
    "city": ["NYC", "LA", "Chicago", "Houston", "NYC"],
    "score": [88.5, 92.1, 79.3, 85.0, 91.2]
}, index=["p1", "p2", "p3", "p4", "p5"])

print(df)

       name  age     city  score
p1    Alice   25      NYC   88.5
p2      Bob   32       LA   92.1
p3  Charlie   18  Chicago   79.3
p4    Diana   47  Houston   85.0
p5     Evan   29      NYC   91.2


### `.loc` — select by LABEL

`.loc[row_label, column_label]` selects using the actual index/column *names*, not their
numeric position. Very important: with `.loc`, slices are **inclusive of the end label**.

In [2]:
print(df.loc["p1"])                 # entire row with index label "p1"
print()
print(df.loc["p1", "name"])          # single cell -> "Alice"
print()
print(df.loc["p1":"p3"])              # rows p1 through p3 INCLUSIVE (unlike Python slicing!)
print()
print(df.loc[:, "name"])              # all rows, just the "name" column
print()
print(df.loc["p1":"p3", ["name", "age"]])   # rows p1-p3, only name & age columns

name     Alice
age         25
city       NYC
score     88.5
Name: p1, dtype: object

Alice

       name  age     city  score
p1    Alice   25      NYC   88.5
p2      Bob   32       LA   92.1
p3  Charlie   18  Chicago   79.3

p1      Alice
p2        Bob
p3    Charlie
p4      Diana
p5       Evan
Name: name, dtype: str

       name  age
p1    Alice   25
p2      Bob   32
p3  Charlie   18


### `.iloc` — select by INTEGER POSITION

`.iloc[row_position, column_position]` works purely by position, like NumPy/list indexing —
slices are **exclusive of the end**, matching standard Python slicing rules.

In [3]:
print(df.iloc[0])              # first row, by position
print()
print(df.iloc[0, 1])            # row 0, column 1 -> age of Alice
print()
print(df.iloc[0:3])              # rows 0, 1, 2 (position 3 EXCLUDED, like normal Python slicing)
print()
print(df.iloc[:, 0])              # all rows, first column
print()
print(df.iloc[0:2, 0:2])          # rows 0-1, columns 0-1

name     Alice
age         25
city       NYC
score     88.5
Name: p1, dtype: object

25

       name  age     city  score
p1    Alice   25      NYC   88.5
p2      Bob   32       LA   92.1
p3  Charlie   18  Chicago   79.3

p1      Alice
p2        Bob
p3    Charlie
p4      Diana
p5       Evan
Name: name, dtype: str

     name  age
p1  Alice   25
p2    Bob   32


### `.loc` vs `.iloc` — side-by-side comparison

| | `.loc` | `.iloc` |
|---|---|---|
| Selects by | Label / name | Integer position |
| Slice `a:b` | **Includes** `b` | **Excludes** `b` |
| Example | `df.loc["p1":"p3"]` | `df.iloc[0:3]` |
| Use when | You know the row/column names | You know positions, or index has no meaningful labels |

### Boolean indexing / filtering — the most-used technique in all of pandas

Build a `True`/`False` mask from a condition, then use it to filter rows — exactly like NumPy
boolean masking, but on labeled data.

In [4]:
mask = df["age"] > 25
print(mask)     # a boolean Series, one value per row

print(df[mask])                  # rows where age > 25
print()
print(df[df["age"] > 25])         # more common: write the condition directly

p1    False
p2     True
p3    False
p4     True
p5     True
Name: age, dtype: bool
     name  age     city  score
p2    Bob   32       LA   92.1
p4  Diana   47  Houston   85.0
p5   Evan   29      NYC   91.2

     name  age     city  score
p2    Bob   32       LA   92.1
p4  Diana   47  Houston   85.0
p5   Evan   29      NYC   91.2


In [5]:
# Combining conditions -- use & (and), | (or), ~ (not); each condition needs parentheses
print(df[(df["age"] > 25) & (df["city"] == "NYC")])
print()
print(df[(df["city"] == "NYC") | (df["city"] == "LA")])

    name  age city  score
p5  Evan   29  NYC   91.2

     name  age city  score
p1  Alice   25  NYC   88.5
p2    Bob   32   LA   92.1
p5   Evan   29  NYC   91.2


In [6]:
# .isin() -- check membership against a list of values, cleaner than chained OR conditions
print(df[df["city"].isin(["NYC", "LA"])])

# .between() -- check a numeric range
print(df[df["age"].between(20, 30)])

     name  age city  score
p1  Alice   25  NYC   88.5
p2    Bob   32   LA   92.1
p5   Evan   29  NYC   91.2
     name  age city  score
p1  Alice   25  NYC   88.5
p5   Evan   29  NYC   91.2


### Combining `.loc` with a boolean condition

`.loc` isn't just for labels — it also accepts a boolean mask, and lets you filter rows AND
pick columns in a single call.

In [7]:
print(df.loc[df["age"] > 25, ["name", "score"]])   # filter rows, select specific columns

     name  score
p2    Bob   92.1
p4  Diana   85.0
p5   Evan   91.2


### Selecting a single value quickly: `.at` and `.iat`

For getting/setting a *single* scalar value, `.at` (label-based) and `.iat` (position-based)
are faster than `.loc`/`.iloc` because they skip some of the general-purpose overhead.

In [8]:
print(df.at["p1", "name"])    # single value by label -- faster equivalent of df.loc["p1","name"]
print(df.iat[0, 0])            # single value by position -- faster equivalent of df.iloc[0,0]

df.at["p1", "age"] = 26         # set a single value directly
print(df.loc["p1"])

Alice
Alice
name     Alice
age         26
city       NYC
score     88.5
Name: p1, dtype: object


### ⚠️ Common pitfall: chained indexing

Writing `df[df["age"] > 25]["score"]` works for *reading*, but assigning through a chain like
`df[df["age"] > 25]["score"] = 100` is unreliable — pandas can't guarantee it modifies the
original DataFrame. Use `.loc` for both filtering AND assigning in one step instead.

In [9]:
df_copy = df.copy()

# ❌ Risky pattern (may trigger a SettingWithCopyWarning, may not actually update df_copy):
# df_copy[df_copy["age"] > 25]["score"] = 100

# ✅ Correct pattern: filter rows and select the column together, inside ONE .loc call
df_copy.loc[df_copy["age"] > 25, "score"] = 100
print(df_copy)

       name  age     city  score
p1    Alice   26      NYC  100.0
p2      Bob   32       LA  100.0
p3  Charlie   18  Chicago   79.3
p4    Diana   47  Houston  100.0
p5     Evan   29      NYC  100.0


### 🧠 Quick check

1. What's the key difference in slice behavior between `.loc["a":"c"]` and `.iloc[0:3]`?
2. Why would you use `.isin(["NYC", "LA"])` instead of `(df["city"]=="NYC") | (df["city"]=="LA")`?
3. What's the safe way to filter rows AND update a column at the same time?

<details>
<summary>Answers</summary>

1. `.loc` slices are **inclusive** of the end label; `.iloc` slices are **exclusive** of the
   end position (standard Python slicing behavior).
2. `.isin()` is more concise and scales cleanly to any number of values, versus chaining many
   `|` conditions by hand.
3. Use a single `.loc[row_condition, column_name] = value` call, rather than chaining two
   separate bracket lookups.
</details>

### ✍️ Practice

1. Using the sample DataFrame, select rows `"p2"` through `"p4"` (inclusive) using `.loc`.
2. Select the same rows by position using `.iloc` (remember the different slice behavior!).
3. Filter for people older than 20 AND with a score above 85.
4. Use `.loc` to set every NYC resident's `score` to `100` in one line.

Continue to **`04_data_inspection_and_cleaning.ipynb`** next.